# Handwired Neural Machines — native Tesla T4 2DOF articulated verification
Runs real Handwired-generated WGSL controller and 2DOF mechanics on a physical NVIDIA GPU. Browser WebGPU, gait learning and 3D locomotion are independent future gates.


In [ ]:
import json,subprocess,traceback,pathlib,time
root=pathlib.Path('/kaggle/working')
report={'ok':False,'stages':[]}
def run(label,cmd):
 print('##',label,flush=True)
 process=subprocess.run(cmd,shell=True,executable='/bin/bash',text=True,
  stdout=subprocess.PIPE,stderr=subprocess.STDOUT,timeout=1400)
 print(process.stdout[-5500:],flush=True)
 report['stages'].append({'name':label,'code':process.returncode,
  'tail':process.stdout[-1200:]})
 (root/'machines-native-job.json').write_text(json.dumps(report,indent=2))
 if process.returncode:raise RuntimeError('Failed: '+label)
try:
 run('Identify physical GPU','nvidia-smi --query-gpu=name,driver_version --format=csv,noheader')
 run('Install Vulkan ICD','apt-get update -qq && apt-get install -y -qq libnvidia-gl-580 libvulkan1 vulkan-tools')
 run('Physical Vulkan adapter','vulkaninfo --summary | grep -E "deviceName|deviceType|driverName" | head -12')
 run('Install wgpu-native and Bun','python -m pip -q install wgpu==0.32.0 && curl -fsSL https://bun.sh/install | bash')
 run('Clone exact Machines branch','git clone --depth 1 --branch feat/frontier-08-machines https://github.com/Chrisbryan17/handwired.git /kaggle/temp/handwired-machines')
 run('Install locked dependencies','cd /kaggle/temp/handwired-machines && /root/.bun/bin/bun install --frozen-lockfile')
 run('Export actual WGSL and independent CPU oracle','cd /kaggle/temp/handwired-machines && /root/.bun/bin/bun scripts/export-machines-native.ts /kaggle/working')
 run('Verify true NVIDIA 2DOF dynamics vs CPU','python /kaggle/temp/handwired-machines/ops/kaggle/native-machines/verify_machines.py /kaggle/working')
 run('Export genuine 3D robotics GPU render shader',
  'cd /kaggle/temp/handwired-machines && /root/.bun/bin/bun scripts/export-machines-render.ts /kaggle/working')
 run('Native physical T4 shader compilation and render-pipeline link',
  'python /kaggle/temp/handwired-machines/ops/kaggle/native-machines/verify_visual_shader.py /kaggle/working')
 report['evidence']=json.loads((root/'machines-native-hardware.json').read_text())
 visual=json.loads((root/'machines-render-compile-report.json').read_text())
 report['visualShaderCompilation']=visual
 report['ok']=report['evidence'].get('ok',False) and visual.get('ok',False)
except Exception as error:
 report['error']=str(error);traceback.print_exc()
finally:
 report['finished']=time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime())
 (root/'machines-native-job.json').write_text(json.dumps(report,indent=2))
print('ARTICULATED_GPU_HARDWARE',report['ok'],flush=True)
